# Silver — Chamados Hora e Canais (`case.silver`)

Notebook de transformação da camada **Bronze → Silver** das tabelas `tb_chamados_hora` e `tb_canais`, seguindo a estrutura: **Setup → Transformação → Debug/validação**.

**Objetivo:** converter horários em timestamps, documentar e sinalizar inconsistências, e padronizar os textos da dimensão Canais. As gravações são determinísticas e usam `overwrite` sobre as mesmas fontes Bronze (idempotência para dados de entrada inalterados).

## Setup

Garante o schema `case.silver` e define os nomes das tabelas utilizados no notebook.

In [0]:
from pyspark.sql import functions as F

catalog = "case"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

bronze_hora = f"{bronze_schema}.tb_chamados_hora"
silver_hora = f"{silver_schema}.tb_chamados_hora"
bronze_chamados = f"{bronze_schema}.tb_chamados"
bronze_canais = f"{bronze_schema}.tb_canais"
silver_canais = f"{silver_schema}.tb_canais"

# Período informado no enunciado: atendimentos de janeiro/2025 a julho/2026 (inclusive).
periodo_inicio = "2025-01-01"
periodo_fim = "2026-07-31"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
print(f"Schema {silver_schema} garantido. Iniciando transformações...")

## 1. Transformação Bronze → Silver — Chamados Hora

A tabela `case.bronze.tb_chamados_hora` possui **131.838 registros** na análise original. Regras:

* **Padronização de colunas:** renomear `ID_Chamado`, `ID_Cliente`, `Hora_Abertura_Chamado`, `Hora_Inicio_Atendimento` e `Hora_Finalizacao_Atendimento` de PascalCase para snake_case (`id_chamado`, `id_cliente`, `hora_abertura_chamado`, `hora_inicio_atendimento`, `hora_finalizacao_atendimento`), alinhando com as demais tabelas Silver. `id_chamado` é convertido para `BIGINT` (mesmo tipo da `tb_chamados`).
* **Datas:** converter as três colunas de horário de texto (formato observado `dd/MM/yyyy ... HH:mm:ss`, com caractere corrompido em “às”) para `timestamp`.
* **Correção de ano 2027 → ano de referência:** o PDF do case declara o período de coleta como janeiro/2025 a julho/2026. Registros com abertura em 2027 são considerados erro de digitação. O ano correto é determinado pela data de **início** do atendimento (ou **finalização** se início for nulo): se a outra data cai no mesmo dia/mês em **2025**, a abertura é corrigida para 2025; se em **2026**, para 2026. Isso preserva a ordem cronológica abertura ≤ início ≤ finalização. Antes esses registros eram apenas sinalizados como `ANO_ABERTURA_SUSPEITO`; agora são corrigidos na origem.
* **Ordem cronológica:** após a correção de ano, classificar apenas ordem inválida (`inicio < abertura` | `fim < inicio` | `fim < abertura`) como `ORDEM_CRONOLOGICA_INVALIDA`; demais como `CONSISTENTE`.
* **Integridade referencial:** verificar se o par `(id_chamado, id_cliente)` existe na Bronze de Chamados. Primeiro, identificar se o chamado existe; depois, se o cliente corresponde. A referência usa `distinct()` para neutralizar duplicidades da Bronze de Chamados.

### Diagnóstico e decisões documentadas

| Verificação (base original) | Resultado |
|---|---:|
| Total | **131.838** |
| Abertura nula | 0 |
| Início nulo | **3.414** (2,59%) |
| Finalização nula | 0 |
| Falhas de conversão dos horários preenchidos | 0 |
| `ID_Chamado` / `ID_Cliente` nulos | 0 / 0 |
| `ID_Chamado` duplicado em Chamados Hora | 0 |
| `ID_Chamado` repetido na Bronze de Chamados | **2.636 IDs** (nenhum associado a clientes distintos naquela base) |
| Pares chamado/cliente sem correspondência na Bronze de Chamados | **1.330** (1,01%) |

**Nota:** as 1.359 comparações divergentes inicialmente observadas foram afetadas pelas duplicidades da Bronze de Chamados. A contagem correta da tabela de horários é de **1.330 pares sem correspondência**. Isso **não** significa que o chamado inexista: o cliente é diferente para um ID existente.


In [0]:
df_bronze_hora = spark.table(bronze_hora)

# Padroniza nomes de colunas: PascalCase → snake_case (alinhamento com as demais tabelas Silver).
df_silver_hora = (
    df_bronze_hora
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento")
    # Alinha id_chamado para BIGINT (mesmo tipo da tb_chamados).
    .withColumn("id_chamado", F.col("id_chamado").cast("bigint"))
)

colunas_horario = [
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento"
]

# Converte datas com texto intermediário potencialmente corrompido (ex.: "29/06/2026 �s 14:12:48").
for coluna in colunas_horario:
    df_silver_hora = df_silver_hora.withColumn(
        coluna,
        F.to_timestamp(
            F.regexp_replace(
                F.col(coluna),
                r"^\s*(\d{2}/\d{2}/\d{4}).*?(\d{2}:\d{2}:\d{2})\s*$",
                r"$1 $2"
            ),
            "dd/MM/yyyy HH:mm:ss"
        )
    )

# Correção de ano: o PDF do case declara o período como janeiro/2025 a julho/2026.
# Registros com abertura em 2027 são erro de digitação. O ano correto é determinado
# pela data de início (ou finalização, se início for nulo): se a outra data cai no
# mesmo dia/mês em 2025, a abertura é corrigida para 2025; se em 2026, para 2026.
# Isso preserva a ordem cronológica (abertura ≤ início ≤ finalização).
_abertura = F.col("hora_abertura_chamado")
_inicio = F.col("hora_inicio_atendimento")
_fim = F.col("hora_finalizacao_atendimento")

_mesmo_dia_mes = lambda a, b: (F.month(a) == F.month(b)) & (F.dayofmonth(a) == F.dayofmonth(b))

# Determina o ano-alvo: prefere início se disponível, senão finalização.
# Se nenhuma referência bater, usa 2026 como fallback.
_ano_alvo = (
    F.when(_inicio.isNotNull() & _mesmo_dia_mes(_abertura, _inicio), F.year(_inicio))
     .when(_fim.isNotNull() & _mesmo_dia_mes(_abertura, _fim), F.year(_fim))
     .otherwise(F.lit(2026))
)

# Corrige a abertura: 2027 → ano-alvo (2025 ou 2026 conforme a outra data).
df_silver_hora = df_silver_hora.withColumn(
    "hora_abertura_chamado",
    F.when(
        F.year(_abertura) == 2027,
        F.add_months(_abertura, (F.lit(2027) - _ano_alvo) * -12)
    ).otherwise(_abertura)
)

# Início e finalização: correção genérica 2027→2026 como rede de segurança
# (na prática não há registros de início/fim em 2027, mas garante consistência).
for coluna in ["hora_inicio_atendimento", "hora_finalizacao_atendimento"]:
    df_silver_hora = df_silver_hora.withColumn(
        coluna,
        F.when(F.year(F.col(coluna)) == 2027, F.add_months(F.col(coluna), -12))
         .otherwise(F.col(coluna))
    )

abertura = F.col("hora_abertura_chamado")
inicio = F.col("hora_inicio_atendimento")
fim = F.col("hora_finalizacao_atendimento")

# Após a correção de ano, avalia apenas a ordem cronológica.
ordem_invalida = (inicio < abertura) | (fim < inicio) | (fim < abertura)

df_silver_hora = df_silver_hora.withColumn(
    "status_consistencia",
    F.when(ordem_invalida, "ORDEM_CRONOLOGICA_INVALIDA")
     .otherwise("CONSISTENTE")
)

# Identificadores distintos da Bronze de Chamados evitam multiplicações no join.
df_pares = spark.table(bronze_chamados).select("id_chamado", "id_cliente").distinct()
df_ids = df_pares.select("id_chamado").distinct()

df_silver_hora = (
    df_silver_hora
    .join(df_ids.withColumn("chamado_existe", F.lit(True)), on="id_chamado", how="left")
    .join(df_pares.withColumn("par_corresponde", F.lit(True)),
          on=["id_chamado", "id_cliente"], how="left")
    .withColumn(
        "status_integridade_cliente",
        F.when(F.col("chamado_existe").isNull(), "CHAMADO_INEXISTENTE")
         .when(F.col("par_corresponde").isNull(), "CLIENTE_DIVERGENTE")
         .otherwise("CORRESPONDENTE")
    )
    .drop("chamado_existe", "par_corresponde")
)

# Grava a versão completa e classificada, mantendo todos os registros da origem.
(
    df_silver_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_hora)
)

print(f"Tabela {silver_hora} criada/atualizada!")
print(f"Bronze: {df_bronze_hora.count():,} linhas")
print(f"Silver: {spark.table(silver_hora).count():,} linhas")

## 2. Debug — Qualidade dos horários e integridade referencial

Validação **após a gravação**. Valores esperados após a correção de ano (2027 → ano de referência 2025/2026) e padronização de colunas:

| `status_consistencia` | Registros (antes) | Registros (agora) |
|---|---:|---:|
| `CONSISTENTE` | 131.639 | **131.837** |
| `ANO_ABERTURA_SUSPEITO` | **198** | *(status removido — registros corrigidos)* |
| `ORDEM_CRONOLOGICA_INVALIDA` | **1** | **1** (inalterado — caso original, não relacionado a 2027) |

| `status_integridade_cliente` | Registros |
|---|---:|
| `CORRESPONDENTE` | 130.508 |
| `CLIENTE_DIVERGENTE` | **1.330** |
| `CHAMADO_INEXISTENTE` | 0 |

`CONSISTENTE` avalia a **ordem cronológica**; não significa que o início esteja preenchido. A coluna `id_chamado` agora é `BIGINT` e os nomes seguem snake_case.

In [0]:
# === DEBUG: qualidade e integridade ===
df_silver_hora = spark.table(silver_hora)

print(f"Total de linhas: {df_silver_hora.count():,}")

print("\nNulos por coluna:")
df_silver_hora.select(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["id_chamado", "id_cliente"] + colunas_horario
]).show(truncate=False)

print("\nStatus cronológico:")
df_silver_hora.groupBy("status_consistencia").count().show(truncate=False)

print("\nIntegridade chamado/cliente:")
df_silver_hora.groupBy("status_integridade_cliente").count().show(truncate=False)

print("\nInterseção das classificações:")
df_silver_hora.groupBy("status_consistencia", "status_integridade_cliente") \
    .count().orderBy("status_consistencia", "status_integridade_cliente").show(truncate=False)

print("\nIDs de chamado duplicados na Silver:")
print(df_silver_hora.groupBy("id_chamado").count().filter(F.col("count") > 1).count())

## 3. Debug — Período previsto no case (jan/2025 a jul/2026)

O enunciado informa que os atendimentos coletados correspondem ao período **01/01/2025 a 31/07/2026**, inclusive. Essa é uma **regra de negócio independente da ordem cronológica**: três timestamps poderiam estar ordenados e ainda assim cair fora do período.

**Resultado observado antes da correção:**

| Ano de abertura | Registros (antes) |
|---|---:|
| 2025 | 71.085 |
| 2026 | 60.555 |
| 2027 | **198** |

**Após a correção 2027 → ano de referência:** os 198 registros de 2027 foram ajustados para o ano da data de início/finalização (2025 ou 2026 conforme o caso), eliminando a categoria 2027.

| Ano de abertura | Registros (antes) | Registros (agora) |
|---|---:|---:|
| 2025 | 71.085 | 71.192 (+107 corrigidos para 2025) |
| 2026 | 60.555 | 60.646 (+91 corrigidos para 2026) |
| 2027 | **198** | **0** |

- **0 aberturas fora do período** (antes eram 198; agora todas estão dentro de jan/2025–jul/2026).
- Máximo de abertura: `2026-07-31` (dentro do período declarado).
- `ORDEM_CRONOLOGICA_INVALIDA`: **1** (caso original, não relacionado à correção de 2027).

In [0]:
# === DEBUG: intervalo temporal previsto no enunciado ===
df_silver_hora = spark.table(silver_hora)
limite_inferior = F.lit(periodo_inicio).cast("date")
limite_superior = F.lit(periodo_fim).cast("date")

print(f"Período esperado: {periodo_inicio} a {periodo_fim} (inclusive)")
for c in colunas_horario:
    fora = (
        F.col(c).isNotNull()
        & ((F.to_date(F.col(c)) < limite_inferior)
           | (F.to_date(F.col(c)) > limite_superior))
    )
    print(f"{c}: {df_silver_hora.filter(fora).count()} registros fora do período")

print("\nMínimos e máximos por coluna:")
df_silver_hora.agg(*(
    [F.min(c).alias(f"min_{c}") for c in colunas_horario]
    + [F.max(c).alias(f"max_{c}") for c in colunas_horario]
)).show(truncate=False)

print("\nDistribuição por ano de abertura:")
df_silver_hora.groupBy(
    F.year("hora_abertura_chamado").alias("ano_abertura")
).count().orderBy("ano_abertura").show()

# Indicador de conferência: não altera a tabela Silver.
fora_em_alguma_coluna = None
for c in colunas_horario:
    cond = F.col(c).isNotNull() & (
        (F.to_date(F.col(c)) < limite_inferior)
        | (F.to_date(F.col(c)) > limite_superior)
    )
    fora_em_alguma_coluna = cond if fora_em_alguma_coluna is None else (fora_em_alguma_coluna | cond)
print("Registros com pelo menos uma data fora do período:",
      df_silver_hora.filter(fora_em_alguma_coluna).count())

## 4. Debug — Durações dos atendimentos (somente análise)

Foram avaliados apenas registros com ordem cronológica consistente **e início preenchido** (**128.231 registros** na análise original).

| Estatística | Espera (min) | Atendimento (min) |
|---|---:|---:|
| Mínimo | 0,00 | 0,30 |
| 25% | 0,00 | 2,07 |
| Mediana | 0,00 | 4,27 |
| 75% | 5,18 | 14,58 |
| Máximo | 103,87 | 150,00 |

**74.939 registros (58,44%)** desse subconjunto têm espera exatamente zero. O padrão foi **documentado, mas não tratado como erro**, pois não há regra no enunciado que proíba espera zero ou estabeleça limites máximos de duração.

In [0]:
# === DEBUG: durações (não persiste colunas extras na Silver) ===
df_duracoes = (
    spark.table(silver_hora)
    .filter(
        (F.col("status_consistencia") == "CONSISTENTE")
        & F.col("hora_inicio_atendimento").isNotNull()
    )
    .withColumn(
        "tempo_espera_min",
        (F.unix_timestamp("hora_inicio_atendimento")
         - F.unix_timestamp("hora_abertura_chamado")) / 60
    )
    .withColumn(
        "tempo_atendimento_min",
        (F.unix_timestamp("hora_finalizacao_atendimento")
         - F.unix_timestamp("hora_inicio_atendimento")) / 60
    )
)

df_duracoes.select("tempo_espera_min", "tempo_atendimento_min") \
    .summary("count", "min", "25%", "50%", "75%", "max").show()

df_duracoes.agg(
    F.count("*").alias("total_analisado"),
    F.sum(F.when(F.col("tempo_espera_min") == 0, 1).otherwise(0)).alias("espera_zero"),
    F.round(F.avg(F.when(F.col("tempo_espera_min") == 0, 1).otherwise(0)) * 100, 2)
     .alias("percentual_espera_zero")
).show()

## 5. Transformação Bronze → Silver — Canais

Tabela de referência com **4 registros** observados na origem:

| `id_canal` | `nome_canal` original | `tipo_canal` | `status_canal` |
|---:|---|---|---|
| 1 | URA | Autosservico | ativo |
| 2 | CHATBOT | Autosservico | ativo |
| 3 | ATENDIMENTO INICIAL | Humano | ativo |
| 4 | ATENDIMENTO ESPECIALIZADO | Humano | ativo |

**Regra:** converter todas as colunas `string` para minúsculas e remover espaços externos (`lower(trim(...))`). Não modificar `id_canal`, `ingestion_datetime` ou a grafia sem acento de `autosservico`.

In [0]:
df_bronze_canais = spark.table(bronze_canais)
df_silver_canais = df_bronze_canais

for coluna, tipo in df_silver_canais.dtypes:
    if tipo == "string":
        df_silver_canais = df_silver_canais.withColumn(
            coluna, F.lower(F.trim(F.col(coluna)))
        )

(
    df_silver_canais.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_canais)
)

print(f"Tabela {silver_canais} criada/atualizada!")
print(f"Bronze: {df_bronze_canais.count()} linhas")
print(f"Silver: {spark.table(silver_canais).count()} linhas")

## 6. Debug — Canais

Conferência visual da tabela pequena após a gravação. Resultado esperado: `ura`, `chatbot`, `atendimento inicial` e `atendimento especializado`, todos com textos minúsculos. Nenhuma validação adicional é necessária para esta etapa.

In [0]:
# === DEBUG: valores da dimensão Canais ===
df_silver_canais = spark.table(silver_canais)
display(df_silver_canais.orderBy("id_canal"))
df_silver_canais.printSchema()